In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Camada Silver — Limpeza e Padronização do Airbnb Rio de Janeiro
# MAGIC
# MAGIC **Objetivo desta etapa:** transformar os dados brutos da Bronze em uma versão limpa,
# MAGIC tipada corretamente e padronizada, pronta para ser consultada com confiança.
# MAGIC
# MAGIC **Decisões tomadas com base na análise de Qualidade de Dados (notebook 02):**
# MAGIC - Registros com `price = 0` são **mantidos**, mas sinalizados em uma coluna de flag —
# MAGIC   a decisão de incluir/excluir da análise final fica para a etapa de Análise.
# MAGIC - Outliers extremos de preço são **mantidos**, também sinalizados por flag, e a limitação
# MAGIC   é documentada no README (não removemos dados automaticamente sem justificativa de negócio).
# MAGIC - Colunas administrativas com altíssima proporção de nulos (~32%) e irrelevantes às perguntas
# MAGIC   de negócio são descartadas nesta camada.
# MAGIC - Colunas de texto longo (descrições, regras da casa, etc.) são descartadas por não serem
# MAGIC   necessárias às análises quantitativas propostas.

# COMMAND ----------

from pyspark.sql.functions import col, regexp_replace, when, to_date, concat, lit, trim, lower, expr

CATALOGO = "mvp_airbnb_rj"
SCHEMA_BRONZE = "bronze"
TABELA_BRONZE = "listings_bronze"
SCHEMA_SILVER = "silver"
TABELA_SILVER = "listings_silver"

df_bronze = spark.table(f"{CATALOGO}.{SCHEMA_BRONZE}.{TABELA_BRONZE}")
print(f"Registros na Bronze: {df_bronze.count()}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Seleção de colunas relevantes
# MAGIC
# MAGIC Das 96 colunas originais (+ 6 de metadados de controle), selecionado apenas as que
# MAGIC alimentam as perguntas de negócio do MVP: preço, localização, características do imóvel,
# MAGIC avaliações e dados do host. Colunas de texto livre (descrições, regras da casa, URLs de
# MAGIC imagem) e colunas administrativas com altíssima proporção de nulos são descartadas aqui.
# MAGIC
# MAGIC **Documentação da decisão:** colunas como `instant_bookable`, `cancellation_policy`,
# MAGIC `require_guest_profile_picture`, `is_business_travel_ready`, `requires_license`, `license`
# MAGIC e `jurisdiction_names` foram descartadas por apresentarem ~32% de nulos e não serem
# MAGIC relevantes às perguntas de negócio definidas no objetivo do MVP.

# COMMAND ----------

colunas_selecionadas = [
    "id", "host_id", "host_name", "host_since", "host_is_superhost",
    "host_response_time", "host_response_rate", "host_listings_count",
    "neighbourhood_cleansed", "neighbourhood_group_cleansed", "city",
    "latitude", "longitude",
    "property_type", "room_type", "accommodates", "bathrooms", "bedrooms", "beds",
    "price", "weekly_price", "monthly_price", "security_deposit", "cleaning_fee",
    "minimum_nights", "maximum_nights",
    "availability_30", "availability_60", "availability_90", "availability_365",
    "number_of_reviews", "first_review", "last_review",
    "review_scores_rating", "review_scores_accuracy", "review_scores_cleanliness",
    "review_scores_checkin", "review_scores_communication", "review_scores_location",
    "review_scores_value", "reviews_per_month",
    "instant_bookable", "cancellation_policy",
    "mes_referencia", "arquivo_origem", "data_ingestao",
]

# Filtra apenas as colunas que realmente existem (segurança contra typos)
colunas_existentes = [c for c in colunas_selecionadas if c in df_bronze.columns]
colunas_faltando = [c for c in colunas_selecionadas if c not in df_bronze.columns]
print("Colunas não encontradas (confira o nome):", colunas_faltando)

df_silver = df_bronze.select(*colunas_existentes)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Padronização de campos monetários (texto → decimal)
# MAGIC
# MAGIC No formato original (Inside Airbnb), campos de valor monetário vêm como texto,
# MAGIC com símbolo de moeda e separador de milhar (ex: "$1,200.00"). Convertendo para
# MAGIC decimal, removendo o símbolo "$" e a vírgula de milhar.

# COMMAND ----------

def limpar_valor_monetario(nome_coluna):
    """Remove símbolo de moeda e separador de milhar, convertendo para double.

    Usamos try_cast (via expr) em vez de .cast() porque o Databricks/Unity Catalog roda
    com modo ANSI SQL ativado por padrão: .cast() lança ERRO ao encontrar um valor que não
    pode ser convertido (ex: string vazia ou valor mal formatado vindo de linhas corrompidas
    no CSV de origem). try_cast(), por outro lado, devolve NULL nesses casos, sem quebrar
    a execução — que é o comportamento que queremos aqui na Silver.

    Toda a limpeza (remover "$" e ",") e a conversão de tipo são feitas em uma única
    expressão SQL, para garantir que o try_cast seja aplicado sobre o valor já limpo.
    """
    return expr(
        f"try_cast(regexp_replace(regexp_replace(string(`{nome_coluna}`), '\\\\$', ''), ',', '') as double)"
    )

colunas_monetarias = ["price", "weekly_price", "monthly_price", "security_deposit", "cleaning_fee"]

for coluna in colunas_monetarias:
    if coluna in df_silver.columns:
        df_silver = df_silver.withColumn(coluna, limpar_valor_monetario(coluna))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Correção de tipos inconsistentes
# MAGIC
# MAGIC Como identificado na análise de qualidade, algumas colunas ficaram com tipo "string"
# MAGIC de forma inconsistente com colunas irmãs (ex: `review_scores_cleanliness` como string,
# MAGIC enquanto `review_scores_rating` já veio como integer). Aqui forçamos a tipagem correta
# MAGIC — valores que não conseguirem ser convertidos viram nulo automaticamente (comportamento
# MAGIC padrão do `cast()` no Spark), o que é aceitável, pois eram inconsistências no dado bruto.

# COMMAND ----------

colunas_para_inteiro = [
    "number_of_reviews", "review_scores_cleanliness", "review_scores_location",
    "review_scores_value", "availability_90", "availability_365",
]

for coluna in colunas_para_inteiro:
    if coluna in df_silver.columns:
        df_silver = df_silver.withColumn(coluna, expr(f"try_cast(`{coluna}` as int)"))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Padronização de campos booleanos (texto "t"/"f" → boolean)

# COMMAND ----------

colunas_booleanas = ["host_is_superhost", "instant_bookable"]

for coluna in colunas_booleanas:
    if coluna in df_silver.columns:
        df_silver = df_silver.withColumn(
            coluna,
            when(lower(trim(col(coluna))) == "t", True)
            .when(lower(trim(col(coluna))) == "f", False)
            .otherwise(None)
        )

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Padronização temporal — mes_referencia como data
# MAGIC
# MAGIC Convertendo `mes_referencia` (formato texto "YYYY-MM") em uma coluna de data real
# MAGIC (primeiro dia do mês), o que facilita ordenação, filtros por período e construção
# MAGIC de gráficos de série temporal nas etapas seguintes.

# COMMAND ----------

df_silver = df_silver.withColumn(
    "data_referencia",
    to_date(concat(col("mes_referencia"), lit("-01")), "yyyy-MM-dd")
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Flags de qualidade — preço zerado e outliers (SEM remoção de dados)
# MAGIC
# MAGIC Conforme decisão documentada no início deste notebook: não removemos automaticamente
# MAGIC registros com preço zerado ou valores extremos. Em vez disso, criamos colunas de flag
# MAGIC para que a etapa de Análise decida, com contexto de negócio, como tratar cada caso.
# MAGIC
# MAGIC O limite de outlier foi definido como o percentil 99 da distribuição de preços
# MAGIC (calculado sobre os dados desta camada).

# COMMAND ----------

# Calcula o percentil 99 de price com alta precisão, usando percentile_approx via SQL.
# NOTA: inicialmente usamos df.approxQuantile("price", [0.99], 0.01), mas o parâmetro de erro
# relativo (0.01 = 1%) é grande demais para uma distribuição tão assimétrica (desvio padrão
# maior que a média) — o resultado veio idêntico ao valor máximo, o que indicava baixa precisão,
# não um dado real. percentile_approx com maior "accuracy" (100.000) dá um resultado muito mais
# confiável, ao custo de um pouco mais de processamento.
percentil_99_price = df_silver.select(
    expr("percentile_approx(price, 0.99, 100000)").alias("p99")
).collect()[0]["p99"]
print(f"Percentil 99 do preço: R$ {percentil_99_price:.2f}")

df_silver = (
    df_silver
    .withColumn("flag_price_zero", when(col("price") == 0, True).otherwise(False))
    .withColumn("flag_price_outlier", when(col("price") > lit(percentil_99_price), True).otherwise(False))
)

qtd_price_zero = df_silver.filter(col("flag_price_zero")).count()
qtd_price_outlier = df_silver.filter(col("flag_price_outlier")).count()
print(f"Registros sinalizados com price = 0: {qtd_price_zero}")
print(f"Registros sinalizados como outlier (price > percentil 99 = R$ {percentil_99_price:.2f}): {qtd_price_outlier}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Padronização de texto — bairro (neighbourhood_cleansed)
# MAGIC
# MAGIC Usado o `neighbourhood_cleansed` (campo já normalizado pelo próprio Inside Airbnb)
# MAGIC em vez do campo `neighbourhood` original (preenchido livremente pelo host, com maior
# MAGIC taxa de nulos e inconsistência de escrita). Aplicamos trim para remover espaços extras.

# COMMAND ----------

if "neighbourhood_cleansed" in df_silver.columns:
    df_silver = df_silver.withColumn("neighbourhood_cleansed", trim(col("neighbourhood_cleansed")))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 8. Persistir a camada Silver como tabela Delta

# COMMAND ----------

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{SCHEMA_SILVER}")

(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOGO}.{SCHEMA_SILVER}.{TABELA_SILVER}")
)

print(f"Tabela Silver criada: {CATALOGO}.{SCHEMA_SILVER}.{TABELA_SILVER}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 9. Validação final

# COMMAND ----------

df_validacao_silver = spark.table(f"{CATALOGO}.{SCHEMA_SILVER}.{TABELA_SILVER}")

print(f"Total de registros na Silver: {df_validacao_silver.count()}")
print(f"Total de colunas na Silver: {len(df_validacao_silver.columns)}")

df_validacao_silver.printSchema()

display(
    df_validacao_silver.select(
        "id", "price", "flag_price_zero", "flag_price_outlier",
        "neighbourhood_cleansed", "room_type", "data_referencia"
    ).limit(10)
)
